# Relevancy, Actor Extraction, & Stance Detection Pipeline (Gemini Unified API)
Project Capstone NLP: Analisis Wacana Kebijakan Transisi Energi (PLTU vs PLTN) di Indonesia.
* **Model**: Google Gemini API (`gemini-3.1-flash-lite`) terstandarisasi untuk seluruh tahap.
* **Ground Truth Benchmark**: `news baru - News fixx.csv` (42 artikel total, 30 relevan, 12 tidak relevan).
* **Arsitektur**: Pipeline 2 Tahap (Stage 1: Relevansi $\rightarrow$ Stage 2: Ekstraksi Aktor & Stance) + Konsolidasi CSV Tunggal.


## 1. Import Libraries & Muat Dataset Input
Memuat dataset langsung dari file input tunggal: `news baru - News fixx.csv`.

In [1]:
import pandas as pd
import numpy as np
import os
import re

# =============================================================================
# FILE INPUT TUNGGAL
# =============================================================================
INPUT_FILE = 'news baru - News fixx.csv'

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"File '{INPUT_FILE}' tidak ditemukan di direktori kerja saat ini.")

# Baca file dengan deteksi encoding otomatis
df_matched = pd.DataFrame()
for enc in ['utf-8', 'latin1', 'cp1252']:
    for sep in [',', ';', None]:
        try:
            df_try = pd.read_csv(INPUT_FILE, sep=sep, engine='python' if sep is None else 'c', encoding=enc)
            if df_try.shape[1] > 1 and len(df_try) > 0:
                df_matched = df_try
                break
        except Exception:
            continue
    if len(df_matched) > 0:
        break

df_matched.columns = df_matched.columns.astype(str).str.strip()
df_annot = df_matched.copy()  # df_annot dan df_matched bersumber dari dataset tunggal ini

print(f"[Sukses] Berhasil memuat dataset dari '{INPUT_FILE}'")
print(f"Total baris: {len(df_matched)} | Total kolom: {df_matched.shape[1]}")
print(f"Daftar kolom: {df_matched.columns.tolist()[:10]}")
display(df_matched.head(3))


[Sukses] Berhasil memuat dataset dari 'news baru - News fixx.csv'
Total baris: 42 | Total kolom: 24
Daftar kolom: ['no', 'source', 'url', 'title', 'date', 'keyword', 'target', 'format', 'content', 'Actor1']


,no,source,url,title,date,keyword,target,format,content,Actor1,...,Evidence2,Actor3,Stance3,Evidence3,Actor4,Stance4,Evidence4,Actor5,Stance5,Evidence5
0,1,detik,https://www.detik.com/bali/bisnis/d-7972473/pl...,"PLTM Santong Hasilkan 3,5 Juta kWh Per Tahun d...",2025-06-19 16:36:00,PLTU,PLTU,article,Pembangkit Listrik Tenaga Mikrohidro (PLTM) Sa...,General Manager (GM) PT Perusahaan Listrik Ne...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,detik,https://news.detik.com/berita/d-6900154/menter...,"Menteri LHK: Setelah Hujan, Kualitas Udara di ...",2023-08-28 16:34:00,PLTU,PLTU,article,"Untuk mengatasi polusi udara di Jakarta, Bogor...",Menteri Lingkungan Hidup dan Kehutanan (LHK) S...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,kompas,https://nasional.kompas.com/read/2020/01/16/10...,YLBHI Sebut Ada 26 Kasus Pencemaran Udara pada...,2020-01-16 10:27:00,PLTU,PLTU,article,"JAKARTA, KOMPAS.com- Yayasan Lembaga Bantuan H...",Ketua Bidang Manajemen PengetahuanYLBHIRahma M...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Validasi & Standarisasi Kolom
Memastikan kolom `title`, `content`, dan `Stance1` tersedia dan bersih untuk dievaluasi.

In [2]:
# Validasi dan standarisasi nama kolom 'title'
def find_column(df, candidates):
    for col in df.columns:
        if col.lower() in candidates:
            return col
    return None

title_col = find_column(df_matched, ['title', 'judul', 'news_title'])
content_col = find_column(df_matched, ['content', 'isi', 'berita', 'article', 'text'])

if title_col and title_col != 'title':
    df_matched['title'] = df_matched[title_col]
    df_annot['title'] = df_annot[title_col]

if content_col and content_col != 'content':
    df_matched['content'] = df_matched[content_col]
    df_annot['content'] = df_annot[content_col]

# Hapus baris dengan judul atau konten kosong
initial_len = len(df_matched)
df_matched = df_matched.dropna(subset=['title']).reset_index(drop=True)
df_annot = df_matched.copy()

print(f"Jumlah artikel siap diproses untuk cek relevansi: {len(df_matched)} / {initial_len}")
display(df_matched[['no', 'title']].head(5))


Jumlah artikel siap diproses untuk cek relevansi: 42 / 42


,no,title
0,1,"PLTM Santong Hasilkan 3,5 Juta kWh Per Tahun d..."
1,2,"Menteri LHK: Setelah Hujan, Kualitas Udara di ..."
2,3,YLBHI Sebut Ada 26 Kasus Pencemaran Udara pada...
3,24,"Cegah Listrik Mati Bergilir, PLTU Dimodifikasi..."
4,4,"Ekspor Batu Bara Dilarang, Ini Kata Adaro"


## 3. Set Up Google GenAI Client & API Key

In [3]:
from google import genai
from google.genai import types
import time
import json
import os

# Ambil API key dari Colab Secrets atau Environment Variables
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('GEMMA') or userdata.get('GEMINI_API_KEY')
except Exception:
    pass

if not api_key:
    api_key = os.environ.get('GEMINI_API_KEY') or os.environ.get('GEMMA')

client = genai.Client(api_key=api_key)
print('Google GenAI Client siap.')


Google GenAI Client siap.


## 4. Kompresi Konteks Ekstraktif (KWIC) & Schema Model Rubrik Tergeneralisasi
* **Broadened Lexical Filter**: Menggunakan pola akar kata energi yang diperluas untuk data skala besar (1.000+ baris) agar tidak terjadi false dismissal.
* **Rubric-Based Prompt**: Tidak menggunakan aturan overfitting kasus kecil (tanpa menyebut nama kasus spesifik). Berbasis rubrik substantif: kebijakan, emisi, dekarbonisasi, pensiun dini, dan pendanaan transisi energi di Indonesia.
* **Structured Output**: `{id: int, relevancy: bool, confidence: float, evidence_quote: str, reasoning: str}`.

In [4]:
from pydantic import BaseModel, Field
from typing import List
import re
import time
from google.genai import types

MODEL_NAME = 'gemini-3.1-flash-lite'

class RelevancyItem(BaseModel):
    id: int = Field(description='ID baris dokumen artikel berita.')
    relevancy: bool = Field(description='True jika artikel membahas PLTU (batubara) atau PLTN (nuklir) di Indonesia; False jika tidak relevan.')
    confidence: float = Field(description='Tingkat keyakinan model antara 0.0 hingga 1.0.', ge=0.0, le=1.0)
    reasoning: str = Field(description='Penjelasan ringkas 1 kalimat mengapa artikel diklasifikasikan relevan atau tidak relevan.')

class RelevancyBatchResponse(BaseModel):
    results: List[RelevancyItem] = Field(description='Daftar hasil klasifikasi relevansi batch.')

RELEVANCY_SYSTEM_INSTRUCTION = (
    "Anda adalah analis kebijakan energi dan infrastruktur kelistrikan Indonesia tingkat pakar.\n"
    "Tugas Anda adalah mengklasifikasikan apakah setiap artikel berita RELEVAN (relevancy = true) atau TIDAK RELEVAN (relevancy = false) terhadap:\n"
    "- PLTU (Pembangkit Listrik Tenaga Uap / batubara) di Indonesia, ATAU\n"
    "- PLTN (Pembangkit Listrik Tenaga Nuklir) di Indonesia.\n\n"
    "PANDUAN KLASIFIKASI:\n"
    "1. RELEVAN (relevancy = true): Artikel membahas operasional, kapasitas, emisi, polusi udara, co-firing biomassa, pasokan batubara, pensiun dini, transisi energi/bauran kelistrikan, atau rencana pembangunan PLTN/reaktor SMR di Indonesia.\n"
    "2. TIDAK RELEVAN (relevancy = false): Peristiwa murni luar negeri tanpa kaitan kebijakan Indonesia, berita kriminal hukum umum di mana PLTU hanya disebut pasif sebagai barang bukti, atau berita lingkungan umum tanpa fokus pembangkit listrik.\n"
)

def evaluate_relevancy_batch(batch_df, client, max_retries=3):
    """Mengevaluasi batch relevansi menggunakan Gemini 3.1 Flash Lite."""
    prompt_parts = ["Analisis relevansi artikel-artikel berita berikut terhadap isu PLTU/PLTN di Indonesia:\n"]
    for idx, row in batch_df.iterrows():
        title = str(row.get('title', ''))
        content = str(row.get('content', ''))[:4500]
        prompt_parts.append(f"--- DOKUMEN ID: {idx} ---\nJudul: {title}\nIsi Ringkas:\n{content}\n")
    full_prompt = "\n".join(prompt_parts)

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=full_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=RELEVANCY_SYSTEM_INSTRUCTION,
                    response_mime_type="application/json",
                    response_schema=RelevancyBatchResponse,
                    temperature=0.0
                )
            )
            parsed = RelevancyBatchResponse.model_validate_json(response.text)
            return parsed.results
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 * (attempt + 1))
            else:
                return [RelevancyItem(id=idx, relevancy=True, confidence=0.5, reasoning='Fallback error') for idx in batch_df.index]

print(f'Stage 1 Relevancy Schema & Gemini API ({MODEL_NAME}) siap.')


Stage 1 Relevancy Schema & Gemini API (gemini-3.1-flash-lite) siap.


## 5. Eksekusi Stateful & Resumable (Dapat Dilanjutkan Kapan Saja)
* **Idempotent Resume**: Jika proses berhenti (misal karena batas kuota API tercapai), saat dijalankan kembali program **otomatis melanjutkan dari baris terakhir** tanpa mengulang dari awal (0 kuota terbuang).
* **Incremental Checkpointing**: Menyimpan hasil prediksi ke disk (`relevancy_checkpoint.csv`) langsung setelah setiap batch selesai.

In [5]:
import os
import pandas as pd
import time

RELEVANCY_CHECKPOINT_FILE = 'relevancy_checkpoint_v2.csv'
batch_size = 5

df_matched['row_id'] = df_matched.index
df_matched['gold_relevancy'] = (df_matched['Stance1'] != 'Irrelevant') & (df_matched['Stance1'].notna())

relevancy_map = {}
if os.path.exists(RELEVANCY_CHECKPOINT_FILE):
    try:
        df_existing = pd.read_csv(RELEVANCY_CHECKPOINT_FILE)
        for _, r in df_existing.iterrows():
            relevancy_map[int(r['row_id'])] = {
                'relevancy': bool(r['relevancy']),
                'confidence': float(r['confidence']),
                'reasoning': str(r['reasoning'])
            }
        print(f"[Resume] Ditemukan checkpoint: {len(relevancy_map)} artikel sudah teranotasi relevansinya.")
    except Exception as e:
        print(f"[Peringatan] Gagal memuat checkpoint relevansi: {e}")

df_rel_to_process = df_matched[~df_matched['row_id'].isin(relevancy_map.keys())].copy()
print(f"Total artikel: {len(df_matched)} | Selesai sebelumnya: {len(relevancy_map)} | Sisa antrean: {len(df_rel_to_process)}")

if len(df_rel_to_process) > 0:
    total_batches = (len(df_rel_to_process) + batch_size - 1) // batch_size
    new_rows = []
    for i in range(0, len(df_rel_to_process), batch_size):
        batch_df = df_rel_to_process.iloc[i:i + batch_size]
        b_idx = (i // batch_size) + 1
        print(f"=== Relevancy Batch {b_idx}/{total_batches} (Row IDs: {batch_df.index.tolist()}) ===")
        results = evaluate_relevancy_batch(batch_df, client)
        for res in results:
            relevancy_map[res.id] = {'relevancy': res.relevancy, 'confidence': res.confidence, 'reasoning': res.reasoning}
            new_rows.append({'row_id': res.id, 'relevancy': res.relevancy, 'confidence': res.confidence, 'reasoning': res.reasoning})
        time.sleep(1.0)

    if new_rows:
        write_hdr = not os.path.exists(RELEVANCY_CHECKPOINT_FILE)
        pd.DataFrame(new_rows).to_csv(RELEVANCY_CHECKPOINT_FILE, mode='a', index=False, header=write_hdr, encoding='utf-8')
        print(f"  [Tersimpan] Berhasil mencatat {len(new_rows)} artikel ke {RELEVANCY_CHECKPOINT_FILE}")

df_matched['pred_relevancy'] = df_matched['row_id'].apply(lambda x: relevancy_map.get(x, {}).get('relevancy', False))
df_matched['relevancy_confidence'] = df_matched['row_id'].apply(lambda x: relevancy_map.get(x, {}).get('confidence', 0.0))
df_matched['relevancy_reasoning'] = df_matched['row_id'].apply(lambda x: relevancy_map.get(x, {}).get('reasoning', ''))
print(f"\nDistribusi Prediksi Relevansi: {df_matched['pred_relevancy'].value_counts().to_dict()}")


[Resume] Ditemukan checkpoint: 42 artikel sudah teranotasi relevansinya.
Total artikel: 42 | Selesai sebelumnya: 42 | Sisa antrean: 0

Distribusi Prediksi Relevansi: {True: 33, False: 9}


## 6. Evaluasi Metrik & Disaggregated Error Analysis
* **Benchmark Ground Truth**: `news baru - News fixx.csv` (`Stance1`).
* **Aturan Ground Truth Biner**:
  * `Stance1` in [`SUPPORT`, `OPPOSE`, `NEUTRAL`] -> **RELEVANT** (`True`).
  * `Stance1` == `Irrelevant` (atau kosong) -> **IRRELEVANT** (`False`).
* **Evaluasi Parsial Otomatis**: Jika proses berhenti sebagian karena kuota, evaluasi tetap dapat berjalan untuk artikel yang sudah selesai di-checkpoint.

In [6]:
from sklearn.metrics import confusion_matrix, classification_report, f1_score, precision_score, recall_score
import pandas as pd

y_true_rel = df_matched['gold_relevancy'].values
y_pred_rel = df_matched['pred_relevancy'].values

cm_rel = confusion_matrix(y_true_rel, y_pred_rel, labels=[False, True])
df_cm_rel = pd.DataFrame(
    cm_rel,
    index=['True IRRELEVANT (0)', 'True RELEVANT (1)'],
    columns=['Pred IRRELEVANT (0)', 'Pred RELEVANT (1)']
)

print('=' * 75)
print('1. CONFUSION MATRIX: RELEVANSI BERITA (STAGE 1)')
print('=' * 75)
display(df_cm_rel)
print('-' * 75)
print(f"Akurasi Relevansi:  {((cm_rel[0,0] + cm_rel[1,1]) / cm_rel.sum())*100:.2f}%")
print(f"Presisi Relevansi:  {precision_score(y_true_rel, y_pred_rel, zero_division=0)*100:.2f}%")
print(f"Recall Relevansi:   {recall_score(y_true_rel, y_pred_rel, zero_division=0)*100:.2f}%")
print(f"Macro-F1 Relevansi: {f1_score(y_true_rel, y_pred_rel, zero_division=0):.4f}")
print('=' * 75)


1. CONFUSION MATRIX: RELEVANSI BERITA (STAGE 1)


,Pred IRRELEVANT (0),Pred RELEVANT (1)
True IRRELEVANT (0),9,3
True RELEVANT (1),0,30


---------------------------------------------------------------------------
Akurasi Relevansi:  92.86%
Presisi Relevansi:  90.91%
Recall Relevansi:   100.00%
Macro-F1 Relevansi: 0.9524


---
# Tahap 2: Target-Attributed Argument Mining & Verbatim Grounding Pipeline

Tahap ini mengekstrak relasi argumentatif lengkap untuk setiap artikel berita relevan:
$$\mathcal{T} = (\text{Aktor}, \text{Peran/Afiliasi}, \text{Target Isu}, \text{Bentuk Tuturan}, \text{Kutipan Verbatim}, \text{Sikap/Stance})$$

**Keunggulan Metodologis vs Pipeline Konvensional**:
1. **Joint Constrained Decoding (Pydantic Schema)**: Menghindari error perambatan (*cascading failure*) dengan mengekstrak relasi aktor-bukti-sikap secara terpadu.
2. **Target-Dependent Stance Classification**: Sikap (Support, Oppose, Neutral) dievaluasi secara spesifik terhadap Target Isu artikel (`PLTU`, `PLTN`, atau `UMUM`).
3. **Deterministic Character Span Resolution**: Menghitung indeks posisi karakter persis `[start_char, end_char]` di teks asli untuk menjamin nol halusinasi (*zero hallucination*).
4. **Greedy Bipartite Matching Evaluation**: Evaluasi berbasis tumpang-tindih token entitas ($IoU \ge 0.40$), Character Span IoU untuk bukti, serta Confusion Matrix & Macro-F1 untuk klasifikasi sikap.

In [7]:
import os
import time
import json
import re
from typing import List, Literal, Optional, Tuple, Dict, Any
from pydantic import BaseModel, Field
import pandas as pd
from google import genai
from google.genai import types

# ==========================================
# 1. Pydantic Structured Output Schema
# ==========================================

class DetectedActor(BaseModel):
    actor_name: str = Field(description="Nama lengkap tokoh atau individu yang dikutip atau menyatakan pandangan.")
    role_or_affiliation: str = Field(description="Jabatan, institusi, afiliasi, atau peran tokoh.")
    target_topic: Literal["PLTU", "PLTN", "UMUM"] = Field(description="Target spesifik yang disikapi oleh tokoh: 'PLTU', 'PLTN', atau 'UMUM'.")
    speech_act_type: Literal["DIRECT_QUOTE", "INDIRECT_SPEECH", "AUTHOR_OPINION"] = Field(description="Bentuk tuturan: 'DIRECT_QUOTE', 'INDIRECT_SPEECH', atau 'AUTHOR_OPINION'.")
    verbatim_evidence: str = Field(description="Kutipan langsung atau pernyataan kunci persis (verbatim quote) dari teks artikel asli.")
    stance: Literal["SUPPORT", "OPPOSE", "NEUTRAL"] = Field(description="Sikap tokoh terhadap target_topic: 'SUPPORT', 'OPPOSE', atau 'NEUTRAL'.")
    rationale: str = Field(description="Penjelasan ringkas 1 kalimat mengapa bukti teks tersebut mencerminkan sikap yang dipilih.")

class ArticleActorDetection(BaseModel):
    id: int = Field(description="ID unik dokumen artikel")
    actors: List[DetectedActor] = Field(default_factory=list, description="Daftar seluruh aktor kunci dan bukti sikapnya dalam artikel ini.")

class BatchActorDetectionResponse(BaseModel):
    results: List[ArticleActorDetection]

# ==========================================
# 2. Generalized Operational Rubric Instruction
# ==========================================

ACTOR_SYSTEM_INSTRUCTION = """Anda adalah sistem Information Extraction (IE) dan Argument Mining tingkat pakar untuk analisis kebijakan publik dan transisi energi di Indonesia (fokus PLTU Batubara vs PLTN Nuklir).
Tugas Anda adalah mengekstrak tokoh/aktor kunci beserta sikap (stance) dan bukti kutipan verbatimnya terhadap Target Fokus isu yang ditentukan untuk setiap dokumen (PLTU, PLTN, atau UMUM).

### PANDUAN OPERASIONAL & ATURAN EKSTRAKSI:
1. **Definisi Aktor**:
   - Tokoh perorangan, pejabat pemerintah, pimpinan BUMN/swasta, pakar/akademisi, aktivis LSM, atau saksi warga yang membuat pernyataan eksplisit terkait energi.
   - **KOLOM OPINI / TAJUK RENCANA**: Jika artikel merupakan opini/analisis (kolom opini), maka **penulis artikel tersebut** (yang namanya dan profilnya tercantum di byline atau akhir teks) adalah **Aktor Utama** dengan `speech_act_type: AUTHOR_OPINION`.
   - **Bukan Aktor**: Nama tokoh yang hanya disebut secara pasif sebagai objek pembicaraan orang lain.
   - **Konsolidasi Per Tokoh**: Untuk setiap tokoh unik, hasilkan tepat 1 entri yang memuat kutipan dan sikap paling representatif terhadap Target Fokus dokumen.

2. **Atribusi Tuturan (Speech Act & Verbatim Evidence)**:
   - `verbatim_evidence` WAJIB merupakan cuplikan teks ASLI yang ada di dalam artikel (substring persis). DILARANG memparafrase.
   - Pisahkan antara orang yang berbicara (Aktor) dan topik yang dibicarakan (Target).

3. **Klasifikasi Sikap (Target-Dependent Stance Classification)**:
   - Evaluasi sikap tokoh SECARA KHUSUS terhadap **Target Fokus Dokumen** yang diberikan:
     * `SUPPORT`: Mendukung eksistensi, investasi, kelanjutan operasi, atau teknologi PLTU/PLTN terkait.
     * `OPPOSE`: Mengkritik, memprotes dampak polusi/lingkungan, menolak proyek, atau mendesak penghentian/pensiun dini.
     * `NEUTRAL`: Bersifat prosedural, netral, regulatori, informatif, atau data faktual tanpa keberpihakan normatif yang jelas.

4. Jika artikel tidak memuat pernyataan sikap dari tokoh perorangan, kembalikan `actors: []`.
"""

# ==========================================
# 3. Deterministic Character Span Resolver
# ==========================================

def resolve_evidence_span(content: str, evidence: str) -> Dict[str, Any]:
    """Menemukan posisi karakter persis [start_char, end_char] dari bukti kutipan di dalam teks artikel."""
    if not evidence or not content:
        return {'start_char': -1, 'end_char': -1, 'is_grounded': False, 'match_type': 'EMPTY'}
    clean_ev = evidence.strip(' "“\'”«»').strip()
    if not clean_ev:
        return {'start_char': -1, 'end_char': -1, 'is_grounded': False, 'match_type': 'EMPTY'}
    idx = content.find(clean_ev)
    if idx != -1:
        return {'start_char': idx, 'end_char': idx + len(clean_ev), 'is_grounded': True, 'match_type': 'EXACT'}
    lower_idx = content.lower().find(clean_ev.lower())
    if lower_idx != -1:
        return {'start_char': lower_idx, 'end_char': lower_idx + len(clean_ev), 'is_grounded': True, 'match_type': 'CASE_INSENSITIVE'}
    normalized_pattern = re.escape(re.sub(r'\s+', ' ', clean_ev))
    pattern = re.sub(r'\\ ', r'\\s+', normalized_pattern)
    match = re.search(pattern, content, re.IGNORECASE)
    if match:
        return {'start_char': match.start(), 'end_char': match.end(), 'is_grounded': True, 'match_type': 'NORMALIZED_WHITESPACE'}
    leading_anchor = clean_ev[:35].strip()
    if len(leading_anchor) >= 15:
        l_idx = content.lower().find(leading_anchor.lower())
        if l_idx != -1:
            return {'start_char': l_idx, 'end_char': min(len(content), l_idx + len(clean_ev)), 'is_grounded': True, 'match_type': 'LEADING_ANCHOR'}
    return {'start_char': -1, 'end_char': -1, 'is_grounded': False, 'match_type': 'HALLUCINATED_OR_PARAPHRASED'}

# ==========================================
# 4. Batch Extraction Function with Fault-Tolerance
# ==========================================

def extract_actors_batch(batch_df: pd.DataFrame, client: genai.Client, model_id: str = 'gemini-3.1-flash-lite', max_retries: int = 3) -> List[ArticleActorDetection]:
    prompt_parts = ["Berikut adalah teks artikel berita. Ekstrak tokoh kunci beserta bukti kutipan verbatim dan sikap (stance) terhadap Target Fokus yang ditentukan:\n"]
    for _, row in batch_df.iterrows():
        doc_id = int(row['no'] if 'no' in row else row['id'])
        target_fokus = str(row.get('target', 'UMUM')).strip()
        title = str(row.get('title', ''))
        content = str(row.get('content', ''))[:8500]
        prompt_parts.append(f"--- DOKUMEN ID: {doc_id} ---\nTarget Fokus Isu: {target_fokus}\nJudul: {title}\nIsi Teks:\n{content}\n")
    full_prompt = "\n".join(prompt_parts)
    base_wait = 2
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=model_id,
                contents=full_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=ACTOR_SYSTEM_INSTRUCTION,
                    response_mime_type="application/json",
                    response_schema=BatchActorDetectionResponse,
                    temperature=0.0,
                )
            )
            parsed = BatchActorDetectionResponse.model_validate_json(response.text)
            return parsed.results
        except Exception as e:
            err_str = str(e).lower()
            if 'quota' in err_str or '429' in err_str or 'resourceexhausted' in err_str:
                print(f"[Quota Limit] API rate limit hit ({e}). Menunggu 10s...")
                time.sleep(10)
            elif '500' in err_str or 'internal' in err_str:
                wait_t = base_wait * (2 ** attempt)
                print(f"[Retry {attempt+1}/{max_retries}] Transient API error: {e}. Backoff {wait_t}s...")
                time.sleep(wait_t)
            else:
                if len(batch_df) > 1:
                    print(f"[Fallback Single] Batch {len(batch_df)} gagal ({e}). Mengurai per artikel...")
                    single_results = []
                    for _, s_row in batch_df.iterrows():
                        single_results.extend(extract_actors_batch(pd.DataFrame([s_row]), client, model_id, max_retries=2))
                    return single_results
                else:
                    print(f"[Skip] Gagal memproses Dokumen {batch_df.iloc[0].get('no', batch_df.iloc[0].get('id'))}: {e}")
                    return [ArticleActorDetection(id=int(batch_df.iloc[0].get('no', batch_df.iloc[0].get('id'))), actors=[])]
    if len(batch_df) > 1:
        single_results = []
        for _, s_row in batch_df.iterrows():
            single_results.extend(extract_actors_batch(pd.DataFrame([s_row]), client, model_id, max_retries=1))
        return single_results
    return [ArticleActorDetection(id=int(batch_df.iloc[0].get('no', batch_df.iloc[0].get('id'))), actors=[])]

print('Pydantic Schema, Deterministic Span Resolver, dan Fungsi Ekstraksi siap.')


Pydantic Schema, Deterministic Span Resolver, dan Fungsi Ekstraksi siap.


## 9. Eksekusi Stateful & Resumable untuk Deteksi Aktor

Sel di bawah ini mengiterasi artikel yang teridentifikasi sebagai **Relevan**:
* **Filter Input**: Mengambil ID artikel dari hasil Stage 1 (`relevancy_checkpoint.csv` dengan `relevancy == True`). Jika dijalankan mandiri, melakukan fallback ke ground truth relevansi (`Stance1 != 'Irrelevant'`).
* **Checkpoint Terus-Menerus**: Menyimpan setiap aktor yang terdeteksi ke `actors_checkpoint.csv` secara inkremental (*append-only*).
* **Crash-Proof & Resumable**: Jika koneksi terputus atau batas kuota harian tercapai, script langsung mencatat baris yang belum selesai dan dapat dilanjutkan tanpa mengulang artikel yang telah diolah.

In [8]:
import os
import pandas as pd
import time

ACTORS_CHECKPOINT_FILE = 'actors_grounded_checkpoint_v2.csv'
actor_batch_size = 2

# Fallback otomatis jika cell dijalankan terpisah (standalone) tanpa run cell sebelumnya
if 'df_matched' not in globals() or 'gold_relevancy' not in df_matched.columns:
    INPUT_FILE = 'news baru - News fixx.csv'
    df_matched = pd.read_csv(INPUT_FILE)
    df_matched = df_matched.dropna(subset=['title']).reset_index(drop=True)
    df_matched['row_id'] = df_matched.index
    df_matched['gold_relevancy'] = (df_matched['Stance1'] != 'Irrelevant') & (df_matched['Stance1'].notna())

# Filter artikel relevan berdasarkan ground truth relevansi
df_candidate_actors = df_matched[df_matched['gold_relevancy']].copy()
print(f"[Input Pool] Ditemukan {len(df_candidate_actors)} artikel relevan untuk diekstrak relasi argumentatifnya.")

already_processed_rows = set()
if os.path.exists(ACTORS_CHECKPOINT_FILE):
    try:
        df_existing = pd.read_csv(ACTORS_CHECKPOINT_FILE)
        if 'row_id' in df_existing.columns:
            already_processed_rows = set(df_existing['row_id'].dropna().astype(int))
            print(f"[Resume] Ditemukan checkpoint: {len(already_processed_rows)} artikel relevan telah selesai diproses.")
    except Exception as e:
        print(f"[Peringatan] Gagal membaca checkpoint: {e}")

df_actors_to_process = df_candidate_actors[~df_candidate_actors['row_id'].isin(already_processed_rows)]
print(f"[Antrean] {len(df_actors_to_process)} artikel relevan yang perlu diproses.\n")

if len(df_actors_to_process) > 0:
    total_batches = (len(df_actors_to_process) + actor_batch_size - 1) // actor_batch_size
    for i in range(0, len(df_actors_to_process), actor_batch_size):
        batch_df = df_actors_to_process.iloc[i:i + actor_batch_size]
        batch_num = (i // actor_batch_size) + 1
        print(f"=== Memproses Batch {batch_num}/{total_batches} (Row IDs: {batch_df.index.tolist()}) ===")

        batch_results = extract_actors_batch(batch_df, client)

        checkpoint_rows = []
        for res in batch_results:
            rid = res.id
            match_art = batch_df.loc[batch_df['row_id'] == rid]
            art_content = str(match_art.iloc[0]['content']) if len(match_art) > 0 else ''

            if not res.actors:
                checkpoint_rows.append({
                    'row_id': rid,
                    'actor_name': '',
                    'role_or_affiliation': '',
                    'target_topic': '',
                    'speech_act_type': '',
                    'verbatim_evidence': '',
                    'start_char': -1,
                    'end_char': -1,
                    'stance': '',
                    'rationale': ''
                })
            else:
                for act in res.actors:
                    span_info = resolve_evidence_span(art_content, act.verbatim_evidence)
                    checkpoint_rows.append({
                        'row_id': rid,
                        'actor_name': act.actor_name,
                        'role_or_affiliation': act.role_or_affiliation,
                        'target_topic': act.target_topic,
                        'speech_act_type': act.speech_act_type,
                        'verbatim_evidence': act.verbatim_evidence,
                        'start_char': span_info['start_char'],
                        'end_char': span_info['end_char'],
                        'stance': act.stance,
                        'rationale': act.rationale
                    })

        if checkpoint_rows:
            df_batch_cp = pd.DataFrame(checkpoint_rows)
            write_header = not os.path.exists(ACTORS_CHECKPOINT_FILE)
            df_batch_cp.to_csv(ACTORS_CHECKPOINT_FILE, mode='a', index=False, header=write_header, encoding='utf-8')
            print(f"  [Tersimpan] Berhasil mencatat {len(checkpoint_rows)} entitas ke {ACTORS_CHECKPOINT_FILE}")

        time.sleep(1.0)

    print("\n[Selesai] Seluruh artikel relevan telah selesai diekstraksi!")

if os.path.exists(ACTORS_CHECKPOINT_FILE):
    df_display = pd.read_csv(ACTORS_CHECKPOINT_FILE)
    print(f"\nTotal baris terdaftar di {ACTORS_CHECKPOINT_FILE}: {len(df_display)} ({df_display['row_id'].nunique()} unik artikel)")
    display(df_display.head(10))


[Input Pool] Ditemukan 30 artikel relevan untuk diekstrak relasi argumentatifnya.
[Resume] Ditemukan checkpoint: 30 artikel relevan telah selesai diproses.
[Antrean] 0 artikel relevan yang perlu diproses.


Total baris terdaftar di actors_grounded_checkpoint_v2.csv: 41 (30 unik artikel)


,row_id,actor_name,role_or_affiliation,target_topic,speech_act_type,verbatim_evidence,start_char,end_char,stance,rationale
0,0,Sri Heny Purwanti,General Manager PT PLN Unit Induk Wilayah NTB,PLTU,INDIRECT_SPEECH,"Saat ini, Heny melanjutkan, total pasokan dari...",1614,1742,NEUTRAL,Tokoh memberikan data faktual mengenai porsi p...
1,1,Siti Nurbaya Bakar,Menteri Lingkungan Hidup dan Kehutanan,PLTU,DIRECT_QUOTE,"Ada pula penindakan terhadap perusahaan, PLTU,...",1180,1401,OPPOSE,Tokoh mengidentifikasi PLTU sebagai entitas pe...
2,2,Rahma Mary,Ketua Bidang Manajemen Pengetahuan YLBHI,PLTU,DIRECT_QUOTE,"Bentuk pelanggaran hak atas hidup, beroperasin...",388,530,OPPOSE,Tokoh mengkritik operasional PLTU karena diang...
3,3,Darmawan Prasodjo,Direktur Utama PLN,PLTU,DIRECT_QUOTE,Nah untuk itu kami melakukan pada adjustment p...,706,835,SUPPORT,Tokoh mendukung keberlanjutan operasional PLTU...
4,4,Mahardika Putranto,Sekretaris Perusahaan PT Adaro Energy Tbk,PLTU,DIRECT_QUOTE,Apalagi mengingat bahwa anak-anak perusahaan t...,977,1164,NEUTRAL,Tokoh menyatakan kepatuhan perusahaan terhadap...
5,5,Bayu Aji Prakoso,Direktur Teknologi salah satu pihak konsorsium,PLTU,DIRECT_QUOTE,"Melalui forum ini, Indonesia dapat memperoleh ...",673,804,SUPPORT,Tokoh mendukung pengembangan teknologi baru un...
6,5,Ari Darmawan Pasek,Profesor Institut Teknologi Bandung (ITB),PLTU,DIRECT_QUOTE,Belakangan ini PLTU batu bara menjadi sorotan ...,1306,1495,SUPPORT,Tokoh mendukung keberlangsungan PLTU dengan sy...
7,6,Takashi Irie,Presiden Direktur PT Bhimasena Power Indonesia...,PLTU,DIRECT_QUOTE,Kegiatan ini menandai dimulainya pekerjaan kon...,316,467,SUPPORT,Tokoh menyatakan dukungan dan optimisme terhad...
8,7,Prabowo Subianto,Presiden Republik Indonesia,PLTU,DIRECT_QUOTE,Kami juga memiliki cadangan panas bumi yang sa...,654,844,OPPOSE,Tokoh secara eksplisit menyatakan rencana kebi...
9,8,Roy Waluyo,Kandidat Doktor Teknik Mesin Universitas Gadja...,PLTN,AUTHOR_OPINION,"Dalam konteks ini, pengembangan Pembangkit Lis...",1463,1615,SUPPORT,Penulis berargumen bahwa PLTN adalah solusi ra...


## 10. Evaluasi Metrik: Kontinjensi Aktor & Confusion Matrix Stance

Evaluasi komprehensif dua komponen utama Stage 2:
1. **Tabel Kontinjensi Deteksi Aktor**: Mengukur True Positives (tokoh yang cocok), False Positives (tokoh tambahan), dan False Negatives (tokoh terlewat).
2. **Confusion Matrix Klasifikasi Stance ($3 \times 3$)**: Mengukur keselarasan sikap (SUPPORT, OPPOSE, NEUTRAL) pada seluruh tokoh yang berhasil dicocokkan.

In [9]:
import pandas as pd
import re
from sklearn.metrics import classification_report, confusion_matrix

# Fallback otomatis jika cell dieksekusi secara terpisah/standalone
import os
if 'ACTORS_CHECKPOINT_FILE' not in globals():
    ACTORS_CHECKPOINT_FILE = 'actors_grounded_checkpoint_v2.csv'

if 'df_matched' not in globals() or 'gold_relevancy' not in df_matched.columns:
    INPUT_FILE = 'news baru - News fixx.csv'
    df_matched = pd.read_csv(INPUT_FILE)
    df_matched = df_matched.dropna(subset=['title']).reset_index(drop=True)
    df_matched['row_id'] = df_matched.index
    df_matched['gold_relevancy'] = (df_matched['Stance1'] != 'Irrelevant') & (df_matched['Stance1'].notna())

def normalize_entity(text: str) -> set:
    """Normalisasi teks entitas tokoh dan peran untuk soft entity matching."""
    stop_words = {
        'dan', 'yang', 'dari', 'pada', 'untuk', 'dengan', 'para', 'sebagai',
        'direktur', 'utama', 'general', 'manager', 'menteri', 'presiden', 'ketua',
        'bidang', 'kepala', 'divisi', 'kementerian', 'badan', 'pusat', 'pt', 'pln',
        'esdm', 'lhk', 'dpr', 'brin', 'ugm', 'uiw', 'uid', 'ntb', 'ntt', 's2jb',
        'penulis', 'opini', 'saksi', 'kandidat', 'doktor', 'tbk', 'indonesia'
    }
    tokens = re.findall(r'[a-zA-Z]{3,}', text.lower())
    clean_tokens = {t for t in tokens if t not in stop_words}
    return clean_tokens if clean_tokens else set(tokens)

def evaluate_actor_and_stance(gold_df: pd.DataFrame, predicted_df: pd.DataFrame):
    """Evaluasi kontinjensi deteksi tokoh dan confusion matrix sikap (tanpa scoring evidence)."""
    gold_relevant = gold_df[gold_df['gold_relevancy']].copy()

    total_gold_actors = 0
    total_pred_actors = 0
    true_positives = 0
    stance_matches = 0

    y_true_stance = []
    y_pred_stance = []
    records = []

    for _, row in gold_relevant.iterrows():
        rid = int(row['row_id'])
        gold_actors = []
        for k in range(1, 6):
            act = row.get(f'Actor{k}')
            st = row.get(f'Stance{k}')
            if pd.notna(act) and str(act).strip():
                gold_actors.append({
                    'slot': k,
                    'actor_text': str(act).strip(),
                    'norm_tokens': normalize_entity(str(act).strip()),
                    'stance': str(st).strip().upper() if pd.notna(st) else ''
                })
        total_gold_actors += len(gold_actors)

        preds = predicted_df[predicted_df['row_id'] == rid]
        pred_actors = []
        for _, p_row in preds.iterrows():
            p_name = str(p_row.get('actor_name', '')).strip()
            if p_name and p_name != 'N/A (Irrelevant / No Actor)' and p_name != 'No Actor Detected':
                p_role = str(p_row.get('role_or_affiliation', '')).strip()
                pred_actors.append({
                    'name': p_name,
                    'role': p_role,
                    'norm_tokens': normalize_entity(f"{p_name} {p_role}"),
                    'stance': str(p_row.get('stance', '')).strip().upper()
                })
        total_pred_actors += len(pred_actors)

        matched_gold_indices = set()
        for p in pred_actors:
            best_match_idx = None
            best_overlap = 0.0
            for g_idx, g in enumerate(gold_actors):
                if g_idx in matched_gold_indices:
                    continue
                intersection = p['norm_tokens'].intersection(g['norm_tokens'])
                overlap_ratio = len(intersection) / max(1, min(len(p['norm_tokens']), len(g['norm_tokens'])))
                if overlap_ratio >= 0.40 and overlap_ratio > best_overlap:
                    best_overlap = overlap_ratio
                    best_match_idx = g_idx

            if best_match_idx is not None:
                matched_gold_indices.add(best_match_idx)
                true_positives += 1
                g_match = gold_actors[best_match_idx]
                st_match = (p['stance'] == g_match['stance'])
                if st_match:
                    stance_matches += 1
                y_true_stance.append(g_match['stance'])
                y_pred_stance.append(p['stance'])
                records.append({
                    'row_id': rid,
                    'status': 'MATCH (TP)',
                    'gold_actor': g_match['actor_text'],
                    'pred_actor': f"{p['name']} ({p['role']})",
                    'gold_stance': g_match['stance'],
                    'pred_stance': p['stance'],
                    'stance_match': st_match
                })
            else:
                records.append({
                    'row_id': rid,
                    'status': 'UNMATCHED (FP)',
                    'gold_actor': '<NONE>',
                    'pred_actor': f"{p['name']} ({p['role']})",
                    'gold_stance': '',
                    'pred_stance': p['stance'],
                    'stance_match': False
                })

        for g_idx, g in enumerate(gold_actors):
            if g_idx not in matched_gold_indices:
                records.append({
                    'row_id': rid,
                    'status': 'MISSED (FN)',
                    'gold_actor': g['actor_text'],
                    'pred_actor': '<MISSED>',
                    'gold_stance': g['stance'],
                    'pred_stance': '',
                    'stance_match': False
                })

    precision = true_positives / max(1, total_pred_actors)
    recall = true_positives / max(1, total_gold_actors)
    f1 = 2 * precision * recall / max(1e-6, precision + recall)
    stance_acc = stance_matches / max(1, true_positives)

    stance_labels = ['SUPPORT', 'OPPOSE', 'NEUTRAL']
    stance_report = classification_report(y_true_stance, y_pred_stance, labels=stance_labels, zero_division=0, output_dict=True)
    conf_mat = confusion_matrix(y_true_stance, y_pred_stance, labels=stance_labels)

    return {
        'total_gold_actors': total_gold_actors,
        'total_pred_actors': total_pred_actors,
        'true_positives': true_positives,
        'false_positives': total_pred_actors - true_positives,
        'false_negatives': total_gold_actors - true_positives,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'stance_accuracy_on_matched': stance_acc,
        'stance_macro_f1': stance_report['macro avg']['f1-score'],
        'confusion_matrix_stance': conf_mat,
        'records_df': pd.DataFrame(records)
    }

if os.path.exists(ACTORS_CHECKPOINT_FILE):
    df_pred_actors = pd.read_csv(ACTORS_CHECKPOINT_FILE)
    eval_results = evaluate_actor_and_stance(df_matched, df_pred_actors)

    print('=' * 75)
    print('2. TABEL KONTINJENSI DETEKSI AKTOR (STAGE 2 - ENTITY EXTRACTION)')
    print('=' * 75)
    df_cm_actor = pd.DataFrame([
        ['True Positives (Matched Actors)', eval_results['true_positives']],
        ['False Positives (Extra Actors Quoted)', eval_results['false_positives']],
        ['False Negatives (Missed Ground Truth Actors)', eval_results['false_negatives']]
    ], columns=['Kategori Deteksi', 'Jumlah Tokoh'])
    display(df_cm_actor)
    print('-' * 75)
    print(f"Presisi Deteksi Aktor: {eval_results['precision']*100:.2f}%")
    print(f"Recall Deteksi Aktor:  {eval_results['recall']*100:.2f}%")
    print(f"Macro-F1 Aktor:        {eval_results['f1']:.4f}")

    print('\n' + '=' * 75)
    print('3. CONFUSION MATRIX: KLASIFIKASI STANCE (STAGE 2 - 3-CLASS POLARITY)')
    print('=' * 75)
    df_cm_stance = pd.DataFrame(
        eval_results['confusion_matrix_stance'],
        index=['True SUPPORT', 'True OPPOSE', 'True NEUTRAL'],
        columns=['Pred SUPPORT', 'Pred OPPOSE', 'Pred NEUTRAL']
    )
    display(df_cm_stance)
    print('-' * 75)
    print(f"Akurasi Sikap pada Tokoh Cocok: {eval_results['stance_accuracy_on_matched']*100:.2f}%")
    print(f"Macro-F1 Sikap:                 {eval_results['stance_macro_f1']:.4f}")
    print('=' * 75)


2. TABEL KONTINJENSI DETEKSI AKTOR (STAGE 2 - ENTITY EXTRACTION)


,Kategori Deteksi,Jumlah Tokoh
0,True Positives (Matched Actors),34
1,False Positives (Extra Actors Quoted),7
2,False Negatives (Missed Ground Truth Actors),4


---------------------------------------------------------------------------
Presisi Deteksi Aktor: 82.93%
Recall Deteksi Aktor:  89.47%
Macro-F1 Aktor:        0.8608

3. CONFUSION MATRIX: KLASIFIKASI STANCE (STAGE 2 - 3-CLASS POLARITY)


,Pred SUPPORT,Pred OPPOSE,Pred NEUTRAL
True SUPPORT,11,0,2
True OPPOSE,1,10,0
True NEUTRAL,2,2,6


---------------------------------------------------------------------------
Akurasi Sikap pada Tokoh Cocok: 79.41%
Macro-F1 Sikap:                 0.7837


## 11. Arsitektur End-to-End Pipeline NLP (Tahap 1 & Tahap 2)

Diagram alur metodologi di bawah menggambarkan integrasi dari artikel mentah hingga ekstraksi entitas aktor dan bukti kutipannya:

In [10]:
import base64
from IPython.display import Image, display

mermaid_full_pipeline = """
graph TD
    A[Dataset Berita Mentah: 'news baru - News fixx.csv'] --> B[Validasi Kolom & Standarisasi ID]

    subgraph STAGE_1 [Tahap 1: Relevancy Classification]
        B --> C{Pemeriksaan Keyword Lokal}
        C -- Zero Energy Keywords --> D[Auto-Label: Irrelevant <br/> $0 Cost & 0 API Calls]
        C -- Candidate Present --> E[Ekstraksi Konteks Salient KWIC <br/> Kompresi 73% Teks]
        E --> F[Pure LLM API Call: gemma-4-26b-a4b-it <br/> Structured Output]
        F --> G[relevancy_checkpoint.csv]
        D --> G
    end

    G --> H{Filter: Relevancy == True}
    H -- False / Irrelevant --> I[Arsip Non-Relevan]

    subgraph STAGE_2 [Tahap 2: Actor Detection & Evidence Extraction]
        H -- True / Relevant --> J[Muat Konten Lengkap Artikel <br/> Pertahankan Byline & Kolom Opini]
        J --> K[Micro-Batching & Quota Handling]
        K --> L[Pure LLM API Call: gemma-4-26b-a4b-it <br/> Pydantic Extraction Rubric]
        L --> M[actors_checkpoint.csv <br/> (actor, role, target, evidence, stance)]
    end

    M --> N[Evaluasi Metrik: <br/> Precision, Recall, Grounding, Stance Acc]
    M --> O[Interactive Web Dashboard / Database Engine]

    style STAGE_1 fill:#e8f0fe,stroke:#4285f4,stroke-width:2px;
    style STAGE_2 fill:#e6f4ea,stroke:#34a853,stroke-width:2px;
    style A fill:#fff3e0,stroke:#f57c00,stroke-width:2px;
    style O fill:#fce8e6,stroke:#ea4335,stroke-width:2px;
"""

graphbytes = mermaid_full_pipeline.encode("utf8")
base64_bytes = base64.b64encode(graphbytes)
base64_string = base64_bytes.decode("ascii")
display(Image(url="https://mermaid.ink/img/" + base64_string))

---
## 12. Konsolidasi Output Pipeline ke Satu File Master CSV
Sel di bawah ini menggabungkan seluruh hasil prediksi dari Tahap 1 (Relevansi) dan Tahap 2 (Aktor & Stance) ke dalam satu file CSV tunggal: `news_pipeline_master_output.csv`.

In [11]:
# =============================================================================
# KONSOLIDASI OUTPUT: GABUNGKAN SELURUH PREDIKSI KE SATU FILE MASTER CSV
# =============================================================================
MASTER_OUTPUT_FILE = 'news_pipeline_master_output.csv'

master_rows = []
df_act_cp = pd.read_csv(ACTORS_CHECKPOINT_FILE) if os.path.exists(ACTORS_CHECKPOINT_FILE) else pd.DataFrame()

for _, row in df_matched.iterrows():
    rid = int(row['row_id'])
    doc_no = row['no']
    title = str(row.get('title', ''))
    keyword = str(row.get('keyword', ''))
    target = str(row.get('target', ''))
    is_rel = bool(row.get('pred_relevancy', False))
    rel_conf = float(row.get('relevancy_confidence', 0.0))
    rel_rsn = str(row.get('relevancy_reasoning', ''))
    gold_rel = bool(row.get('gold_relevancy', False))

    doc_actors = df_act_cp[df_act_cp['row_id'] == rid] if not df_act_cp.empty else pd.DataFrame()

    if not is_rel or doc_actors.empty or doc_actors['actor_name'].dropna().str.strip().eq('').all():
        master_rows.append({
            'row_id': rid,
            'no': doc_no,
            'title': title,
            'keyword': keyword,
            'target': target,
            'is_relevant_pred': is_rel,
            'relevancy_confidence': rel_conf,
            'relevancy_reasoning': rel_rsn,
            'is_relevant_gold': gold_rel,
            'actor_name': '',
            'role_or_affiliation': '',
            'target_topic': '',
            'speech_act_type': '',
            'verbatim_evidence': '',
            'start_char': None,
            'end_char': None,
            'stance': '',
            'stance_rationale': ''
        })
    else:
        for _, act in doc_actors.iterrows():
            if pd.isna(act.get('actor_name')) or not str(act.get('actor_name')).strip():
                continue
            master_rows.append({
                'row_id': rid,
                'no': doc_no,
                'title': title,
                'keyword': keyword,
                'target': target,
                'is_relevant_pred': is_rel,
                'relevancy_confidence': rel_conf,
                'relevancy_reasoning': rel_rsn,
                'is_relevant_gold': gold_rel,
                'actor_name': str(act.get('actor_name', '')).strip(),
                'role_or_affiliation': str(act.get('role_or_affiliation', '')).strip(),
                'target_topic': str(act.get('target_topic', '')).strip(),
                'speech_act_type': str(act.get('speech_act_type', '')).strip(),
                'verbatim_evidence': str(act.get('verbatim_evidence', '')).strip(),
                'start_char': int(act.get('start_char', -1)),
                'end_char': int(act.get('end_char', -1)),
                'stance': str(act.get('stance', '')).strip().upper(),
                'stance_rationale': str(act.get('rationale', '')).strip()
            })

df_master = pd.DataFrame(master_rows)
df_master.to_csv(MASTER_OUTPUT_FILE, index=False, encoding='utf-8')
print(f"Master CSV tunggal berhasil disimpan ke: {MASTER_OUTPUT_FILE}")
print(f"Total baris tercatat: {len(df_master)} (mencakup seluruh {len(df_matched)} artikel input)")
display(df_master.head(10))


Master CSV tunggal berhasil disimpan ke: news_pipeline_master_output.csv
Total baris tercatat: 53 (mencakup seluruh 42 artikel input)


,row_id,no,title,keyword,target,is_relevant_pred,relevancy_confidence,relevancy_reasoning,is_relevant_gold,actor_name,role_or_affiliation,target_topic,speech_act_type,verbatim_evidence,start_char,end_char,stance,stance_rationale
0,0,1,"PLTM Santong Hasilkan 3,5 Juta kWh Per Tahun d...",PLTU,PLTU,True,0.95,Artikel membahas bauran energi di NTB yang men...,True,Sri Heny Purwanti,General Manager PT PLN Unit Induk Wilayah NTB,PLTU,INDIRECT_SPEECH,"Saat ini, Heny melanjutkan, total pasokan dari...",1614.0,1742.0,NEUTRAL,Tokoh memberikan data faktual mengenai porsi p...
1,1,2,"Menteri LHK: Setelah Hujan, Kualitas Udara di ...",PLTU,PLTU,True,0.98,Artikel membahas penindakan terhadap PLTU seba...,True,Siti Nurbaya Bakar,Menteri Lingkungan Hidup dan Kehutanan,PLTU,DIRECT_QUOTE,"Ada pula penindakan terhadap perusahaan, PLTU,...",1180.0,1401.0,OPPOSE,Tokoh mengidentifikasi PLTU sebagai entitas pe...
2,2,3,YLBHI Sebut Ada 26 Kasus Pencemaran Udara pada...,PLTU,PLTU,True,1.00,Artikel secara spesifik membahas dampak lingku...,True,Rahma Mary,Ketua Bidang Manajemen Pengetahuan YLBHI,PLTU,DIRECT_QUOTE,"Bentuk pelanggaran hak atas hidup, beroperasin...",388.0,530.0,OPPOSE,Tokoh mengkritik operasional PLTU karena diang...
3,3,24,"Cegah Listrik Mati Bergilir, PLTU Dimodifikasi...",PLTU,PLTU,True,1.00,Artikel membahas modifikasi teknis PLTU di Ind...,True,Darmawan Prasodjo,Direktur Utama PLN,PLTU,DIRECT_QUOTE,Nah untuk itu kami melakukan pada adjustment p...,706.0,835.0,SUPPORT,Tokoh mendukung keberlanjutan operasional PLTU...
4,4,4,"Ekspor Batu Bara Dilarang, Ini Kata Adaro",PLTU batu bara,PLTU,True,1.00,Artikel membahas kebijakan pemerintah terkait ...,True,Mahardika Putranto,Sekretaris Perusahaan PT Adaro Energy Tbk,PLTU,DIRECT_QUOTE,Apalagi mengingat bahwa anak-anak perusahaan t...,977.0,1164.0,NEUTRAL,Tokoh menyatakan kepatuhan perusahaan terhadap...
5,5,44,"Dorong Energi Bersih, Konsorsium Inovasi PLTU ...",PLTU,PLTU,True,1.00,Artikel membahas inovasi teknologi PLTU batuba...,True,Bayu Aji Prakoso,Direktur Teknologi salah satu pihak konsorsium,PLTU,DIRECT_QUOTE,"Melalui forum ini, Indonesia dapat memperoleh ...",673.0,804.0,SUPPORT,Tokoh mendukung pengembangan teknologi baru un...
6,5,44,"Dorong Energi Bersih, Konsorsium Inovasi PLTU ...",PLTU,PLTU,True,1.00,Artikel membahas inovasi teknologi PLTU batuba...,True,Ari Darmawan Pasek,Profesor Institut Teknologi Bandung (ITB),PLTU,DIRECT_QUOTE,Belakangan ini PLTU batu bara menjadi sorotan ...,1306.0,1495.0,SUPPORT,Tokoh mendukung keberlangsungan PLTU dengan sy...
7,6,51,"Struktur Baja ""Boiler"" PLTU Batang Tembus 30 P...",PLTU,PLTU,True,1.00,Artikel membahas progres konstruksi proyek PLT...,True,Takashi Irie,Presiden Direktur PT Bhimasena Power Indonesia...,PLTU,DIRECT_QUOTE,Kegiatan ini menandai dimulainya pekerjaan kon...,316.0,467.0,SUPPORT,Tokoh menyatakan dukungan dan optimisme terhad...
8,7,5,Prabowo Akan Hentikan Operasional PLTU 15 Tahu...,PLTU,PLTU,True,1.00,Artikel membahas kebijakan pemerintah Indonesi...,True,Prabowo Subianto,Presiden Republik Indonesia,PLTU,DIRECT_QUOTE,Kami juga memiliki cadangan panas bumi yang sa...,654.0,844.0,OPPOSE,Tokoh secara eksplisit menyatakan rencana kebi...
9,8,54,Nuklir Sebagai Energi Masa Depan,PLTN,PLTN,True,1.00,Artikel membahas urgensi dan potensi pengemban...,True,Roy Waluyo,Kandidat Doktor Teknik Mesin Universitas Gadja...,PLTN,AUTHOR_OPINION,"Dalam konteks ini, pengembangan Pembangkit Lis...",1463.0,1615.0,SUPPORT,Penulis berargumen bahwa PLTN adalah solusi ra...
